# 계류장 합의 표시 + YOLO26x 학습 (Colab GPU)

지난번(`autolabel_apron.ipynb`)은 Grounding DINO 한 모델이 박스를 그려서, 들판·지붕을 차량으로 그린 실수까지 YOLO가 배웠습니다.
이번에는 **여러 모델이 각자 박스를 그리고, 2개 이상이 같은 자리에 동의한 박스만** 정답으로 씁니다.

| 그리는 모델 | 비행기 | 탑승교 | 지상 차량 | 사람 |
|---|---|---|---|---|
| Grounding DINO (base) | O | O | O | O |
| OWLv2 (large) | O | O | O | O |
| Gemini (키가 있을 때) | O | O | O | O |
| YOLO26x (COCO) | O | | | O |

그 뒤 차량 박스를 한 번 더 거릅니다: 박스 안이 대부분 풀밭(초록)이면 버리고, 비행기보다 큰 '차량'도 버립니다.
Gemini는 학습 사진에 박스를 그릴 때만 쓰고, 화면에서는 학습된 YOLO만 돌아갑니다.

**실행**: 런타임 유형 A100 → `런타임 → 모두 실행`. 처음에 Roboflow 키와 Gemini 키를 묻습니다(없으면 Enter).
**결과**: 마지막 셀이 `apron_consensus.zip`(가중치·학습 기록·표시 견본 그림·출처표)을 내려받습니다. 안 받아지면 왼쪽 📁에서 직접 받습니다.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install ultralytics==8.4.215 roboflow "transformers>=4.51" timm google-genai
from getpass import getpass
RF_KEY = getpass("Roboflow Private API Key (CCTV 데이터를 안 섞으려면 Enter): ").strip()
GEMINI_KEY = getpass("Gemini API Key (Gemini 투표를 안 쓰려면 Enter): ").strip()

## 0. 구글 드라이브 저장본 (있으면 표시 단계를 건너뜀)

In [ ]:
# 구글 드라이브에 지난 표시 결과가 있으면 받아 와서 사진 받기·표시 단계(1, 4, 5, 6)를 건너뛴다.
# 다시 표시하고 싶으면 드라이브의 apron_cache 폴더를 지우거나 이름을 바꾼다.
import shutil
from pathlib import Path
from google.colab import drive
from IPython.core.magic import register_cell_magic
drive.mount("/content/drive")
CACHE = Path("/content/drive/MyDrive/apron_cache")
SAMPLES = Path("/content/label_samples")
CACHED = (CACHE / "auto" / "data.yaml").exists()
if CACHED:
    shutil.copytree(CACHE / "auto", "/content/auto", dirs_exist_ok=True)
    shutil.copytree(CACHE / "label_samples", SAMPLES, dirs_exist_ok=True)
    for f in ["attribution.csv", "label_stats.json"]:
        shutil.copy(CACHE / f, "/content/" + f)
    print("드라이브 저장본 사용: 사진 받기·표시를 건너뜁니다.")
else:
    print("드라이브 저장본 없음: 처음부터 표시합니다.")

@register_cell_magic
def skip_if(line, cell):
    if eval(line, get_ipython().user_ns):
        print("건너뜀 (드라이브 저장본 사용)")
        return
    get_ipython().run_cell(cell)


## 1. 위키미디어 공용에서 공항 항공 사진 받기 (검색 + 분류, 막히면 기다렸다 다시)

In [ ]:
%%skip_if CACHED
import csv, re, time, hashlib, json
from pathlib import Path
import requests

RAW = Path("/content/commons"); RAW.mkdir(parents=True, exist_ok=True)
S = requests.Session()
S.headers["User-Agent"] = "apron-dataset-builder/0.2 (student research project)"
API = "https://commons.wikimedia.org/w/api.php"
QUERIES = ["airport apron aerial", "aerial view airport terminal aircraft", "airport aerial photograph",
           "aircraft parked apron aerial view", "Flughafen Vorfeld Luftbild", "aéroport vue aérienne avions",
           "airport from above planes gates", "aerial photo international airport terminal", "airport apron from above",
           "Luftaufnahme Flughafen Terminal", "aeropuerto vista aérea", "airport tarmac aerial", "airport ramp aerial view",
           "aerial view of airport gates", "aircraft stands aerial", "Flughafen Luftbild Flugzeuge"]
CATEGORIES = ["Category:Aerial photographs of airports", "Category:Airport aprons"]
OK_LICENSE = re.compile(r"^(cc0|cc[- ]by(-sa)?[- ]?[0-9.]*|public domain|pd.*)$", re.I)
MAX_IMAGES = 700

def get(params, tries=6):
    for k in range(tries):
        try:
            r = S.get(API, params={**params, "maxlag": 5}, timeout=40)
            if r.status_code == 200 and "json" in r.headers.get("content-type", ""):
                j = r.json()
                if j.get("error", {}).get("code") != "maxlag":
                    return j
        except Exception:
            pass
        time.sleep(2 ** k)          # 1, 2, 4, 8, 16, 32 초 기다렸다 다시
    return {}

meta, seen = [], set()
def add_pages(pages):
    for pg in pages.values():
        ii = (pg.get("imageinfo") or [{}])[0]
        em = ii.get("extmetadata", {})
        lic = em.get("LicenseShortName", {}).get("value", "").strip()
        if pg.get("title") in seen or ii.get("mime") not in ("image/jpeg", "image/png") or ii.get("width", 0) < 1200:
            continue
        if not OK_LICENSE.match(lic) or "nc" in lic.lower() or "nd" in lic.lower():
            continue
        seen.add(pg["title"])
        artist = re.sub("<[^>]+>", "", em.get("Artist", {}).get("value", ""))[:200]
        meta.append(dict(title=pg["title"], url=ii.get("thumburl") or ii["url"], page=ii.get("descriptionurl", ""),
                         license=lic, artist=artist))

II = dict(prop="imageinfo", iiprop="url|extmetadata|size|mime", iiurlwidth=2048)
for q in QUERIES:
    for off in range(0, 500, 50):
        j = get(dict(action="query", format="json", generator="search", gsrsearch=f"{q} filetype:bitmap",
                     gsrnamespace=6, gsrlimit=50, gsroffset=off, **II))
        pages = j.get("query", {}).get("pages", {})
        if not pages:
            break
        add_pages(pages); time.sleep(0.5)
    print(f"검색 {q!r}: 누적 {len(meta)}", flush=True)

# 분류(카테고리): 하위 분류 1단계까지 파일 이름을 모은 뒤 50개씩 정보 조회
titles, cats = [], list(CATEGORIES)
for depth in range(2):
    nxt = []
    for c in cats[:300]:
        cont = {}
        while True:
            j = get(dict(action="query", format="json", list="categorymembers", cmtitle=c, cmtype="file|subcat",
                         cmlimit=500, **cont))
            for m in j.get("query", {}).get("categorymembers", []):
                (nxt if m["ns"] == 14 else titles).append(m["title"])
            if "continue" not in j:
                break
            cont = j["continue"]
        time.sleep(0.3)
    cats = nxt
titles = [t for t in dict.fromkeys(titles) if t not in seen][:4000]
for i in range(0, len(titles), 50):
    j = get(dict(action="query", format="json", titles="|".join(titles[i:i + 50]), **II))
    add_pages(j.get("query", {}).get("pages", {})); time.sleep(0.5)
print("후보", len(meta), "장 (분류에서", len(titles), "개 확인)")

saved = []
for m in meta[:MAX_IMAGES]:
    name = hashlib.md5(m["title"].encode()).hexdigest()[:12] + ".jpg"
    for k in range(4):
        try:
            r = S.get(m["url"], timeout=60)
            if r.status_code == 429:
                time.sleep(5 * (k + 1)); continue
            if r.ok and len(r.content) > 50_000:
                (RAW / name).write_bytes(r.content); m["file"] = name; saved.append(m)
            break
        except Exception:
            time.sleep(3)
    time.sleep(0.2)
with open("/content/attribution.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["file", "title", "license", "artist", "page", "url"]); w.writeheader(); w.writerows(saved)
print("받은 사진", len(saved), "장")

## 2. 테스트 사진 폴더 (선택, 학습에는 쓰지 않음)

In [ ]:
from pathlib import Path
TEST = Path("/content/test"); TEST.mkdir(exist_ok=True)
# 테스트 사진은 노트북에 넣지 않는다(저작권). 비교하려면 왼쪽 📁 에서 /content/test 에 올린다. 본 비교는 내 컴퓨터에서 화면과 같은 방식으로 한다.
print(sorted(p.name for p in TEST.iterdir()))

## 3. 박스 그리는 모델들

조각(1024px, 20% 겹침) + 전체 사진으로 찾고 같은 종류끼리 겹친 박스를 합칩니다.
합의로 거르기 때문에 모델마다 기준을 지난번보다 낮게 잡아 많이 찾게 합니다.

In [ ]:
import torch, numpy as np, cv2
from PIL import Image
from torchvision.ops import batched_nms, box_iou
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, Owlv2Processor, Owlv2ForObjectDetection
from ultralytics import YOLO

DEV = "cuda"
CLASSES = ["aircraft", "jet_bridge", "ground_vehicle", "person"]

def to_class(label):
    l = str(label).lower()
    if "bridge" in l: return 1
    if "plane" in l or "aircraft" in l: return 0
    if "person" in l or "worker" in l: return 3
    if "vehicle" in l or "truck" in l or "car" in l or "cart" in l or "tug" in l or "bus" in l: return 2
    return None

gd_proc = AutoProcessor.from_pretrained("IDEA-Research/grounding-dino-base")
gd = AutoModelForZeroShotObjectDetection.from_pretrained("IDEA-Research/grounding-dino-base").to(DEV).eval()
GD_TEXT = "airplane. jet bridge. vehicle. truck. person."

ow_proc = Owlv2Processor.from_pretrained("google/owlv2-large-patch14-ensemble")
ow = Owlv2ForObjectDetection.from_pretrained("google/owlv2-large-patch14-ensemble").to(DEV).eval()
OW_QUERIES = ["a photo of an airplane", "a jet bridge at an airport gate", "an airport ground vehicle",
              "a small truck or tug on airport tarmac", "a person"]
OW_CLASS = [0, 1, 2, 2, 3]

coco = YOLO("yolo26x.pt")

@torch.no_grad()
def gd_detect(img, thr=0.25):
    inp = gd_proc(images=img, text=GD_TEXT, return_tensors="pt").to(DEV)
    out = gd(**inp)
    try:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, threshold=thr, text_threshold=0.2,
                                                           target_sizes=[img.size[::-1]])[0]
    except TypeError:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, box_threshold=thr, text_threshold=0.2,
                                                           target_sizes=[img.size[::-1]])[0]
    labels = r.get("text_labels", r["labels"])
    rows = [(*b.tolist(), float(s), to_class(l)) for b, s, l in zip(r["boxes"], r["scores"], labels)]
    return [x for x in rows if x[5] is not None]

@torch.no_grad()
def ow_detect(img, thr=0.15):
    inp = ow_proc(text=[OW_QUERIES], images=img, return_tensors="pt").to(DEV)
    out = ow(**inp)
    s = max(img.size)   # OWLv2 는 사진을 정사각형으로 채워 넣는다
    post = getattr(ow_proc, "post_process_grounded_object_detection", None) or ow_proc.post_process_object_detection
    r = post(out, threshold=thr, target_sizes=torch.tensor([[s, s]], device=DEV))[0]
    return [(*b.tolist(), float(sc), OW_CLASS[int(l)]) for b, sc, l in zip(r["boxes"], r["scores"], r["labels"])]

def coco_detect(img):
    r = coco.predict(img, imgsz=1024, conf=0.25, classes=[4, 0], verbose=False)[0]
    return [(*b[:4], b[4], 0 if int(b[5]) == 4 else 3) for b in r.boxes.data.tolist()]

def tiled(detect, img, tile=1024, overlap=0.2):
    W, H = img.size
    rows = list(detect(img))
    step = int(tile * (1 - overlap))
    if max(W, H) > tile * 1.25:
        xs = list(range(0, max(W - tile, 0) + 1, step)); ys = list(range(0, max(H - tile, 0) + 1, step))
        if xs[-1] + tile < W: xs.append(W - tile)
        if ys[-1] + tile < H: ys.append(H - tile)
        for y in ys:
            for x in xs:
                for x1, y1, x2, y2, sc, c in detect(img.crop((x, y, min(x + tile, W), min(y + tile, H)))):
                    rows.append((x1 + x, y1 + y, x2 + x, y2 + y, sc, c))
    if not rows:
        return np.zeros((0, 6))
    t = torch.tensor(rows, dtype=torch.float32)
    t = t[((t[:, 2] - t[:, 0]) * (t[:, 3] - t[:, 1])) < 0.4 * W * H]   # 사진 대부분을 덮는 박스는 버림
    keep = batched_nms(t[:, :4], t[:, 4], t[:, 5].long(), 0.5)
    return t[keep].numpy()
print("모델 준비 완료")

## 4. Gemini 투표 (키가 있을 때만)

사진 한 장에 한 번 묻고 결과를 `/content/gem_cache`에 저장합니다(다시 실행해도 다시 묻지 않음).
무료 등급 한도(429)에 걸리면 다음 모델로 넘어가고, 모두 막히면 그 사진은 나머지 모델끼리만 합의합니다.

In [ ]:
%%skip_if CACHED
from concurrent.futures import ThreadPoolExecutor
import io
GEM_MODELS = ["gemini-3.8-flash", "gemini-3.7-flash", "gemini-3.6-flash", "gemini-3.1-flash-lite"]
GEM_PROMPT = '''Detect every instance of these objects in this airport photo:
- airplane
- jet_bridge (passenger boarding bridge attached to the terminal)
- ground_vehicle (any car, bus, van, truck, aircraft tug/pushback tractor, baggage cart train, fuel truck,
  catering truck, belt loader, stairs truck, container loader). Small vehicles may be only a few pixels; include them.
- person
Do NOT label buildings, roofs, road markings, grass, shadows, or parts of airplanes as vehicles.
Return only a JSON list: [{"label": "airplane|jet_bridge|ground_vehicle|person", "box_2d": [ymin, xmin, ymax, xmax]}]
with coordinates normalised to 0-1000. Return [] if nothing is found.'''
GEM_CACHE = Path("/content/gem_cache"); GEM_CACHE.mkdir(exist_ok=True)
dead = set()
gem_stats = {}

def gem_detect(path):
    cache = GEM_CACHE / (path.stem + ".json")
    if cache.exists():
        return json.loads(cache.read_text())
    if not GEMINI_KEY:
        return None
    from google import genai
    from google.genai import types
    client = genai.Client(api_key=GEMINI_KEY)
    img = Image.open(path).convert("RGB"); W, H = img.size
    small = img.copy(); small.thumbnail((2048, 2048))
    buf = io.BytesIO(); small.save(buf, "JPEG", quality=90)
    for model in GEM_MODELS:
        if model in dead:
            continue
        for k in range(3):
            try:
                r = client.models.generate_content(model=model, contents=[
                        types.Part.from_bytes(data=buf.getvalue(), mime_type="image/jpeg"), GEM_PROMPT],
                    config=types.GenerateContentConfig(response_mime_type="application/json", temperature=0))
                t = r.text.strip(); items = json.loads(t[t.find("["):t.rfind("]") + 1] or "[]")
                rows = []
                for it in items:
                    c = to_class(it.get("label", "")); b = it.get("box_2d")
                    if c is None or not b or len(b) != 4: continue
                    y1, x1, y2, x2 = [float(v) / 1000 for v in b]
                    rows.append([x1 * W, y1 * H, x2 * W, y2 * H, 0.5, c])
                cache.write_text(json.dumps(rows)); gem_stats[model] = gem_stats.get(model, 0) + 1
                return rows
            except Exception as e:
                msg = str(e)
                if "429" in msg or "RESOURCE_EXHAUSTED" in msg:
                    if "PerDay" in msg or "per day" in msg.lower() or k == 2:
                        dead.add(model); break
                    time.sleep(20 * (k + 1))
                elif "503" in msg or "UNAVAILABLE" in msg:
                    time.sleep(10 * (k + 1))
                else:
                    break
    return None

files = sorted(RAW.glob("*.jpg"))
if GEMINI_KEY:
    with ThreadPoolExecutor(4) as ex:
        res = list(ex.map(gem_detect, files))
    print("Gemini 표시한 사진", sum(r is not None for r in res), "/", len(files), "모델별", gem_stats, "막힌 모델", dead)
else:
    print("Gemini 키 없음: 나머지 모델끼리만 합의")

## 5. 합의 표시

같은 종류에서 서로 다른 모델 2개 이상이 같은 자리(IoU 0.3 이상, 또는 작은 박스 중심이 큰 박스 안이고 크기 차이 4배 미만)에 박스를 그렸으면 정답으로 씁니다.
박스는 동의한 모델 박스들의 평균입니다. 그 뒤 차량 박스를 거릅니다.

In [ ]:
%%skip_if CACHED
import shutil, random

def same_place(a, b):
    ia = box_iou(torch.tensor([a[:4]]), torch.tensor([b[:4]]))[0, 0].item()
    if ia >= 0.3:
        return True
    area = lambda r: max(r[2] - r[0], 1) * max(r[3] - r[1], 1)
    s, l = (a, b) if area(a) < area(b) else (b, a)
    cx, cy = (s[0] + s[2]) / 2, (s[1] + s[3]) / 2
    return l[0] <= cx <= l[2] and l[1] <= cy <= l[3] and area(l) / area(s) < 4

def consensus(votes, min_votes=2):
    out = []
    for c in range(4):
        items = [(t, r) for t, arr in votes.items() for r in arr if int(r[5]) == c]
        items.sort(key=lambda x: -x[1][4])
        used = [False] * len(items)
        for i, (t, r) in enumerate(items):
            if used[i]: continue
            used[i] = True; group = [r]; ts = {t}
            for j in range(i + 1, len(items)):
                t2, r2 = items[j]
                if used[j] or t2 in ts: continue
                if same_place(r, r2):
                    used[j] = True; group.append(r2); ts.add(t2)
            if len(ts) >= min_votes:
                out.append((*np.mean([g[:4] for g in group], 0), len(ts), c))
    return np.array(out, dtype=float).reshape(-1, 6)

def green_frac(arr, b):
    x1, y1, x2, y2 = [int(v) for v in b[:4]]
    crop = arr[max(y1, 0):y2, max(x1, 0):x2]
    if crop.size == 0: return 1.0
    hsv = cv2.cvtColor(crop, cv2.COLOR_RGB2HSV)
    g = (hsv[..., 0] >= 30) & (hsv[..., 0] <= 90) & (hsv[..., 1] > 50) & (hsv[..., 2] > 40)
    return g.mean()

def filter_vehicles(b, arr, W, H, stats):
    if not len(b): return b
    area = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    air = area[b[:, 5] == 0]
    keep = []
    for k, r in enumerate(b):
        if int(r[5]) == 2:
            if area[k] > 0.015 * W * H or (len(air) and area[k] > 0.6 * np.median(air)):
                stats["차량_너무큼"] += 1; continue
            if green_frac(arr, r) > 0.3:
                stats["차량_풀밭"] += 1; continue
        keep.append(k)
    return b[keep]

OUT = Path("/content/auto"); shutil.rmtree(OUT, ignore_errors=True)
SAMPLES = Path("/content/label_samples"); shutil.rmtree(SAMPLES, ignore_errors=True); SAMPLES.mkdir()
COLORS = [(255, 60, 60), (60, 160, 255), (255, 200, 0), (0, 220, 120)]
stats = {"차량_너무큼": 0, "차량_풀밭": 0}
teacher_tot = {}
files = sorted(RAW.glob("*.jpg")); random.seed(0); random.shuffle(files)
kept = []
t0 = time.time()
for i, p in enumerate(files):
    try:
        img = Image.open(p).convert("RGB")
    except Exception:
        continue
    W, H = img.size; arr = np.array(img)
    votes = {"gdino": tiled(gd_detect, img), "owlv2": tiled(ow_detect, img), "coco": tiled(coco_detect, img)}
    g = gem_detect(p) if GEMINI_KEY else None
    if g:
        votes["gemini"] = np.array(g, dtype=float).reshape(-1, 6)
    for t, a in votes.items():
        teacher_tot.setdefault(t, np.zeros(4, int))
        teacher_tot[t] += np.bincount(a[:, 5].astype(int), minlength=4) if len(a) else 0
    b = filter_vehicles(consensus(votes), arr, W, H, stats)
    cnt = np.bincount(b[:, 5].astype(int), minlength=4) if len(b) else np.zeros(4, int)
    if cnt[0] < 2 and cnt[1] < 1:        # 비행기 2대 미만이고 탑승교도 없으면 공항 계류장 사진이 아니라고 보고 버림
        continue
    kept.append((p, (W, H), b))
    if len(kept) <= 24:                  # 표시 견본 그림 (내려받아 눈으로 확인)
        a2 = arr.copy()
        for x1, y1, x2, y2, s, c in b:
            cv2.rectangle(a2, (int(x1), int(y1)), (int(x2), int(y2)), COLORS[int(c)], 3)
        im = Image.fromarray(a2); im.thumbnail((1600, 1600)); im.save(SAMPLES / p.name, quality=85)
    if i % 25 == 0:
        print(f"{i}/{len(files)} 남긴 사진 {len(kept)}  {time.time() - t0:.0f}초", flush=True)

n_val = max(int(len(kept) * 0.15), 1)
for j, (p, (W, H), b) in enumerate(kept):
    split = "valid" if j < n_val else "train"
    (OUT / split / "images").mkdir(parents=True, exist_ok=True); (OUT / split / "labels").mkdir(parents=True, exist_ok=True)
    shutil.copy(p, OUT / split / "images" / p.name)
    lines = [f"{int(c)} {(x1 + x2) / 2 / W:.6f} {(y1 + y2) / 2 / H:.6f} {(x2 - x1) / W:.6f} {(y2 - y1) / H:.6f}"
             for x1, y1, x2, y2, s, c in b]
    (OUT / split / "labels" / (p.stem + ".txt")).write_text("\n".join(lines))
tot = np.bincount(np.concatenate([k[2][:, 5].astype(int) for k in kept]), minlength=4) if kept else np.zeros(4, int)
label_stats = {"사진": len(files), "남긴_사진": len(kept), "검증": n_val, "합의_박스": dict(zip(CLASSES, tot.tolist())),
               "모델별_박스": {t: dict(zip(CLASSES, v.tolist())) for t, v in teacher_tot.items()}, "거른_차량": stats}
Path("/content/label_stats.json").write_text(json.dumps(label_stats, ensure_ascii=False, indent=1))
print(json.dumps(label_stats, ensure_ascii=False, indent=1))

## 6. (선택) 사람이 표시한 Roboflow CCTV 사진 섞기

In [ ]:
%%skip_if CACHED
import yaml
if RF_KEY:
    from roboflow import Roboflow
    rf = Roboflow(api_key=RF_KEY)
    MAP = {"plane": 0, "aircraft": 0, "jet bridge": 1, "boarding bridge": 1, "passenger": 3, "staff": 3, "person": 3}
    VEH = {"car", "truck", "bus", "passenger bus", "fire truck", "water service", "baggage truck", "container transport",
           "belt loader", "fuel truck", "pushback truck", "aircraft tractor", "boarding stairway", "aircraft ladder",
           "catering truck", "lift truck", "container loader", "gpu"}
    for ws, name in [("datalabeling-0nprc", "handling"), ("airport-azgu2", "airport_90")]:
        proj = rf.workspace(ws).project(name)
        ver = max(int(str(v.version).split("/")[-1]) for v in proj.versions())
        root = Path(proj.version(ver).download("yolov8", location=f"/content/raw_rf/{name}").location)
        names = yaml.safe_load((root / "data.yaml").read_text())["names"]
        names = list(names.values()) if isinstance(names, dict) else names
        remap = {i: (MAP.get(n.lower()) if n.lower() not in VEH else 2) for i, n in enumerate(names)}
        for split in ("train", "valid"):
            for img in (root / split / "images").glob("*"):
                lab = root / split / "labels" / (img.stem + ".txt")
                rows = []
                for line in (lab.read_text().splitlines() if lab.exists() else []):
                    q = line.split()
                    if q and remap.get(int(q[0])) is not None:
                        rows.append(" ".join([str(remap[int(q[0])])] + q[1:]))
                (OUT / split / "images").mkdir(parents=True, exist_ok=True); (OUT / split / "labels").mkdir(parents=True, exist_ok=True)
                shutil.copy(img, OUT / split / "images" / f"rf_{name}_{img.name}")
                (OUT / split / "labels" / f"rf_{name}_{img.stem}.txt").write_text("\n".join(rows))
        print(name, "추가")
(OUT / "data.yaml").write_text(yaml.safe_dump({"path": str(OUT), "train": "train/images", "val": "valid/images",
                                               "names": dict(enumerate(CLASSES))}))
for s in ("train", "valid"):
    print(s, len(list((OUT / s / "images").glob("*"))), "장")

## 6-1. 표시 결과를 드라이브에 저장

In [ ]:
# 표시 결과를 드라이브에 저장해 다음 실행에서 다시 쓴다.
if not CACHED:
    shutil.rmtree(CACHE, ignore_errors=True)
    shutil.copytree("/content/auto", CACHE / "auto")
    shutil.copytree(SAMPLES, CACHE / "label_samples")
    for f in ["attribution.csv", "label_stats.json"]:
        shutil.copy("/content/" + f, CACHE / f)
    print("드라이브에 저장:", CACHE)


## 7. YOLO26x 학습 (이미지 1280). 표시 모델은 메모리에서 내린다

In [ ]:
import gc
for v in ("gd", "ow"): globals().pop(v, None)
gc.collect(); torch.cuda.empty_cache()
model = YOLO("yolo26x.pt")
model.train(data="/content/auto/data.yaml", epochs=200, patience=40, imgsz=1280, batch=-1, cos_lr=True,
            close_mosaic=15, degrees=5, scale=0.5, fliplr=0.5, project="/content/runs/cons", name="yolo26x",
            exist_ok=True, seed=0, deterministic=True, plots=True)

## 8. 테스트 사진 개수 비교 (글자로 출력)

In [ ]:
new = YOLO("/content/runs/cons/yolo26x/weights/best.pt")
TRUTH = {"munich.jpg": "비행기 7", "dubai.jpg": "비행기 18", "test1.jpg": "비행기 수십"}
for p in sorted(TEST.glob("*.jpg")):
    img = Image.open(p).convert("RGB")
    c1 = np.bincount([int(r[5]) for r in coco_detect(img)], minlength=4)
    r = new.predict(img, imgsz=1280, conf=0.25, verbose=False)[0]
    c2 = np.bincount(r.boxes.cls.int().tolist(), minlength=4)
    print(f"{p.name:11s} 실제 {TRUTH.get(p.name, '?'):8s} | COCO 비행기 {c1[0]:3d} 사람 {c1[3]:3d} | "
          f"새 모델 비행기 {c2[0]:3d} 탑승교 {c2[1]:3d} 차량 {c2[2]:3d} 사람 {c2[3]:3d}")

## 9. 결과 묶어서 내려받기 (안 받아지면 왼쪽 📁 파일에서 `apron_consensus.zip` 직접 다운로드)

In [ ]:
from google.colab import files
R = Path("/content/runs/cons/yolo26x")
for f in ["/content/attribution.csv", "/content/label_stats.json"]:
    shutil.copy(f, R / Path(f).name)
shutil.copytree(SAMPLES, R / "label_samples", dirs_exist_ok=True)
shutil.rmtree(R / "weights_last", ignore_errors=True)
(R / "weights" / "last.pt").unlink(missing_ok=True)      # 같은 크기 파일 하나 줄이기
shutil.make_archive("/content/apron_consensus", "zip", R)
print(round(Path("/content/apron_consensus.zip").stat().st_size / 1e6), "MB")
files.download("/content/apron_consensus.zip")
shutil.copy("/content/apron_consensus.zip", CACHE / "apron_consensus.zip")  # 내려받기가 안 될 때를 대비해 드라이브에도 둔다
